In [11]:
import rasterio
import numpy as np
from scipy.optimize import curve_fit
import pandas as pd

## load in the data

# List of file paths for each year's raster
yearly_files = ['H:/My Drive/NDVI_Exports/step7_NDVI_2001.tif', 'H:/My Drive/NDVI_Exports/step7_NDVI_2001.tif']  # Add all your file paths here

# Initialize a list to store the data
yearly_data = []

# Read each raster
for file in yearly_files:
    with rasterio.open(file) as src:
        data = src.read(1)  # Reading the first band
        yearly_data.append(data)

# Convert to a numpy array with each year's data as a separate layer
stacked_data = np.stack(yearly_data, axis=0)

# Check shape and data
print("Stacked Data Shape:", stacked_data.shape)

Stacked Data Shape: (2, 457, 881)


In [13]:
## deal with NAs
stacked_data = np.nan_to_num(stacked_data, nan=0.0)

## get the min and max values                        
pixel_min = np.min(stacked_data, axis=0)
pixel_max = np.max(stacked_data, axis=0)

print("Pixel-wise Min NDVI:", pixel_min)
print("Pixel-wise Max NDVI:", pixel_max)

# Avoid division by zero by setting min=max to some default range
pixel_range = pixel_max - pixel_min
pixel_range[pixel_range == 0] = 1  # Prevent division by zero

print("Pixel-wise Range:", pixel_range)

# Scale the NDVI values pixel-wise to [0, 1]
scaled_stacked_data = (stacked_data - pixel_min) / pixel_range

# Check the scaled data
print("Scaled NDVI Min:", np.min(scaled_stacked_data))
print("Scaled NDVI Max:", np.max(scaled_stacked_data))



Pixel-wise Min NDVI: [[0.         0.         0.         ... 0.         0.         0.        ]
 [0.2371607  0.2371607  0.2371607  ... 0.34105933 0.34105933 0.34105933]
 [0.2371607  0.2371607  0.2371607  ... 0.34105933 0.34105933 0.34105933]
 ...
 [0.22697142 0.22697142 0.22697142 ... 0.17408991 0.17408991 0.17408991]
 [0.22697142 0.22697142 0.22697142 ... 0.17408991 0.17408991 0.17408991]
 [0.22725512 0.22562444 0.22562444 ... 0.17633677 0.17633677 0.17633677]]
Pixel-wise Max NDVI: [[0.         0.         0.         ... 0.         0.         0.        ]
 [0.2371607  0.2371607  0.2371607  ... 0.34105933 0.34105933 0.34105933]
 [0.2371607  0.2371607  0.2371607  ... 0.34105933 0.34105933 0.34105933]
 ...
 [0.22697142 0.22697142 0.22697142 ... 0.17408991 0.17408991 0.17408991]
 [0.22697142 0.22697142 0.22697142 ... 0.17408991 0.17408991 0.17408991]
 [0.22725512 0.22562444 0.22562444 ... 0.17633677 0.17633677 0.17633677]]
Pixel-wise Range: [[1. 1. 1. ... 1. 1. 1.]
 [1. 1. 1. ... 1. 1. 1.]
 [

In [14]:
def dlc_model(time, xmidS, scalS, xmidA, scalA):
    return (1 / (1 + np.exp((xmidS - time) / scalS))) - (1 / (1 + np.exp((xmidA - time) / scalA)))


In [15]:
# Define the years corresponding to each layer in the stacked_data
start_year = 2001
end_year = 2002
# Define the years corresponding to each layer in the stacked_data
years = range(start_year, end_year + 1)

def fit_dlc(scaled_stacked_data, years):
    dlc_params = []

    for i, year in enumerate(years):
        print(f"Processing year: {year}")

        # Extract NDVI data for the year
        ndvi_values = scaled_stacked_data[i].flatten()  # Flatten to 1D

        # Generate day of the year for each pixel
        time = np.tile(np.arange(1, 366), (ndvi_values.size // 365))

        # If there were less than 4 pixels with NDVI values in the entire sequence, skip fitting
        if len(ndvi_values) < 4:
            dlc_params.append([np.nan] * 4)
            continue
        
        try:
            # Fit the curves, with bounds on the parameters
            popt, _ = curve_fit(dlc_model, time, ndvi_values, bounds=([30, 1, 167, 1], [228, 30, 288, 35]))
            dlc_params.append(popt.tolist())
        except Exception as e:
            print(f"Error fitting year {year}: {e}")
            dlc_params.append([np.nan] * 4)
    
    return dlc_params

# Fit DLC curves
dlc_params = fit_dlc(scaled_stacked_data, years)


Processing year: 2001
Error fitting year 2001: operands could not be broadcast together with shapes (402595,) (402617,) 
Processing year: 2002
Error fitting year 2002: operands could not be broadcast together with shapes (402595,) (402617,) 
